# Chronic Kidney Disease Prediction

Dataset: Kaggle CKD Disease

---

In [16]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

os.makedirs('../models', exist_ok=True)
os.makedirs('../data', exist_ok=True)
os.makedirs('../results', exist_ok=True)

In [17]:

# 1. DATA LOADING

try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

# Project root is one level up from notebooks/ when running from the notebook file
if os.path.basename(SCRIPT_DIR) == 'notebooks':
    PROJECT_ROOT = os.path.dirname(SCRIPT_DIR)
else:
    PROJECT_ROOT = SCRIPT_DIR

def load_kidney_data():
    paths_to_check = [
        # Relative to project root (ML_Module/)
        os.path.join(PROJECT_ROOT, 'data', 'kidney.csv'),
        # Relative to script (notebooks/)
        os.path.join(SCRIPT_DIR, 'data', 'kidney.csv'),
        # Relative to CWD
        '../data/kidney.csv',
    ]
    
    for path in paths_to_check:
        abs_path = os.path.abspath(path)
        exists = os.path.exists(abs_path)

        if exists:
            print(f"\n[Data] Loading from: {abs_path}")
            return pd.read_csv(abs_path)

# Load data
df = load_kidney_data()
print(f"\n[1] Data Loaded: {df.shape[0]} samples, {df.shape[1]-1} features")


[Data] Loading from: c:\Users\www\Desktop\project\ml\data\kidney.csv

[1] Data Loaded: 400 samples, 25 features


In [18]:
# 2. DATA CLEANING & ENCODING
# Identify target column
target_col = None
for col in df.columns:
    if df[col].dtype == 'object':
        unique_vals = set(str(v).lower().strip() for v in df[col].dropna().unique())
        if any(x in unique_vals for x in ['ckd', 'notckd', 'yes', 'no']):
            target_col = col
            break

if target_col is None:
    target_col = df.columns[-1]

print(f"\n[2] Target column: '{target_col}'")
print(f"    Unique values: {df[target_col].unique()}")

# Encode target
target_map = {'ckd': 1, 'notckd': 0, 'yes': 1, 'no': 0, '1': 1, '0': 0}
df[target_col] = df[target_col].astype(str).str.strip().str.lower().map(target_map)
df = df.dropna(subset=[target_col])
df = df.rename(columns={target_col: 'target'})

print(f"    Target Distribution: {dict(df['target'].value_counts().sort_index())}")

# Encode categorical features
for col in df.columns:
    if df[col].dtype == 'object' and col != 'target':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

# 3. MISSING VALUE HANDLING
print(f"\n[3] Missing Values Before deletion: {df.isnull().sum().sum()}")
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
print(f"    Missing Values After deletion: {df.isnull().sum().sum()}")

# 4. FEATURES
print(f"\n[4] Available columns: {list(df.columns)}")


[2] Target column: 'htn'
    Unique values: ['yes' 'no' nan]
    Target Distribution: {0.0: np.int64(251), 1.0: np.int64(147)}

[3] Missing Values Before deletion: 470
    Missing Values After deletion: 0

[4] Available columns: ['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'target', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


In [19]:
# 5. SCALING & SPLIT

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n[5] Train/Test Split: {len(X_train)} / {len(X_test)}")

joblib.dump(scaler, '../models/kidney_scaler.pkl')
joblib.dump(list(X.columns), '../models/kidney_feature_names.pkl')
print("    Scaler saved: ../models/kidney_scaler.pkl")


[5] Train/Test Split: 318 / 80
    Scaler saved: ../models/kidney_scaler.pkl


In [20]:
# 6. MODEL TRAINING

results = []

def evaluate_model(name, model, X_test, y_test, is_torch=False):
    if is_torch:
        model.eval()
        with torch.no_grad():
            outputs = model(torch.FloatTensor(X_test))
            y_pred = (outputs.squeeze() > 0.5).float().numpy()
            y_prob = outputs.squeeze().numpy()
    else:
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_prob)

    print(f"\n{name} Results:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc:.4f}")
    print(f"  Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

    return {
        'Disease': 'Kidney',
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'ROC-AUC': round(roc, 4)
    }

print("\n" + "=" * 60)
print("TRAINING MODELS")
print("=" * 60)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
results.append(evaluate_model('Logistic Regression', lr, X_test_scaled, y_test))

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
results.append(evaluate_model('Random Forest', rf, X_test_scaled, y_test))

# PyTorch MLP
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(X_train_scaled.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train.values).unsqueeze(1))
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

print("\nTraining PyTorch MLP...")
model.train()
for epoch in range(200):
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1}/200, Loss: {epoch_loss/len(train_loader):.4f}")

results.append(evaluate_model('MLP', model, X_test_scaled, y_test, is_torch=True))


TRAINING MODELS

Logistic Regression Results:
  Accuracy:  0.8250
  Precision: 0.7667
  Recall:    0.7667
  F1-Score:  0.7667
  ROC-AUC:   0.9280
  Confusion Matrix:
[[43  7]
 [ 7 23]]

Random Forest Results:
  Accuracy:  0.8750
  Precision: 0.7778
  Recall:    0.9333
  F1-Score:  0.8485
  ROC-AUC:   0.9583
  Confusion Matrix:
[[42  8]
 [ 2 28]]

Training PyTorch MLP...
  Epoch 50/200, Loss: 0.0854
  Epoch 100/200, Loss: 0.0209
  Epoch 150/200, Loss: 0.0204
  Epoch 200/200, Loss: 0.0146

MLP Results:
  Accuracy:  0.8250
  Precision: 0.7857
  Recall:    0.7333
  F1-Score:  0.7586
  ROC-AUC:   0.9140
  Confusion Matrix:
[[44  6]
 [ 8 22]]


In [21]:
# 7. EXPORT BEST MODEL
print("\n" + "=" * 60)
print("MODEL COMPARISON & EXPORT")
print("=" * 60)

results_df = pd.DataFrame(results)
print("\n", results_df.to_string(index=False))

best_idx = results_df['ROC-AUC'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f"\nBest Model: {best_model_name} (ROC-AUC: {results_df.loc[best_idx, 'ROC-AUC']})")

if best_model_name == 'Logistic Regression':
    joblib.dump(lr, '../models/kidney_model.pkl')
elif best_model_name == 'Random Forest':
    joblib.dump(rf, '../models/kidney_model.pkl')
else:
    torch.save({
        'input_dim': X_train_scaled.shape[1],
        'state_dict': model.state_dict()
    }, '../models/kidney_model.pth')

print(f"    Best model saved to ../models/kidney_model.*")
results_df.to_csv('../results/kidney_results.csv', index=False)


MODEL COMPARISON & EXPORT

 Disease               Model  Accuracy  Precision  Recall     F1  ROC-AUC
 Kidney Logistic Regression     0.825     0.7667  0.7667 0.7667   0.9280
 Kidney       Random Forest     0.875     0.7778  0.9333 0.8485   0.9583
 Kidney                 MLP     0.825     0.7857  0.7333 0.7586   0.9140

Best Model: Random Forest (ROC-AUC: 0.9583)
    Best model saved to ../models/kidney_model.*
